# Real Pipeline Dry Run (SPY/TLT/GLD Stub)

This is my scratchpad for graduating the toy AR(1) notebook into the actual repo plumbing. It wires `src.features.build` → `src.models.har_rv` → `src.backtest.engine` on a tiny ETF slice so I can make sure every hand-off behaves before scaling things up.

## Plan
1. Load the TFT-ready CSVs so I’m working with the same targets/features as the main pipeline.
2. Repackage them into the simple parquet artifact that the HAR script expects.
3. Call the official HAR helper to spit out a prediction CSV.
4. Push those preds through the lightweight backtest engine to confirm the workflow.
5. Jot down quick diagnostics and note what’s still missing (multi-horizon, real P&L, etc.).

### Wire up the sandbox
Before I poke at data I import the few libraries I rely on, hop out of the `notebooks/` directory if needed, and define the helper paths (raw, processed, experiments, etc.) so every later step knows where to read and write.

In [9]:
import json
import pathlib
import sys
import numpy as np
import pandas as pd
from datetime import datetime

ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
EXPERIMENTS = ROOT / "experiments"
DATA_DIR = ROOT / "data"
TFT_READY_CANDIDATES = {
    "train": DATA_DIR / "tft_ready_train.csv",
    "val": DATA_DIR / "tft_ready_val.csv",
    "test": DATA_DIR / "tft_ready_test.csv",
    "dataset": DATA_DIR / "tft_ready_dataset.csv",
}
print("Project root:", ROOT)
print("Raw dir:", DATA_RAW)
print("Processed dir:", DATA_PROCESSED)

Project root: /Users/sonalilonkar/Desktop/Projects/DL_Project/vol-forecasting
Raw dir: /Users/sonalilonkar/Desktop/Projects/DL_Project/vol-forecasting/data/raw
Processed dir: /Users/sonalilonkar/Desktop/Projects/DL_Project/vol-forecasting/data/processed


## 1. Inspect TFT-ready inputs
I always start by reusing the TFT-ready CSVs from the main pipeline so the dry run doesn’t drift. This cell loads whichever split files exist (or the combined dataset), adds a tiny `split_hint`, and sorts everything by `(ticker, date)` so the data looks exactly like the real pipeline feeds.

In [10]:
def load_tft_ready():
    split_files = {k: p for k, p in TFT_READY_CANDIDATES.items() if k in ("train","val","test") and p.exists()}
    if split_files:
        frames = []
        for split_name, path in split_files.items():
            part = pd.read_csv(path, low_memory=False)
            part["split_hint"] = split_name
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)
        print(f"Loaded {len(frames)} TFT-ready split files.")
    elif TFT_READY_CANDIDATES["dataset"].exists():
        df = pd.read_csv(TFT_READY_CANDIDATES["dataset"], low_memory=False)
        df["split_hint"] = "dataset"
        print("Loaded data/tft_ready_dataset.csv")
    else:
        raise FileNotFoundError("Place tft_ready_{train,val,test}.csv or tft_ready_dataset.csv under data/.")
    if "asset" in df.columns and "ticker" not in df.columns:
        df = df.rename(columns={"asset": "ticker"})
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker","date"]).reset_index(drop=True)
    return df

tft_df = load_tft_ready()
print(tft_df["ticker"].value_counts().head())
tft_df.head()

Loaded 3 TFT-ready split files.
ticker
GLD    3491
QQQ    3491
SPY    3491
TLT    3491
Name: count, dtype: int64


,date,ticker,logvol_t,target_logvol_t+1,target_logvol_t+5,target_logvol_t+22,weekday,is_month_end,is_opex,is_holiday_eve,...,roll_mean_vol_132d,roll_std_vol_132d,roll_mean_ret_252d,roll_std_ret_252d,roll_mean_vol_252d,roll_std_vol_252d,pca_vol_cs_1,pca_vol_cs_2,pca_vol_cs_3,split_hint
0,2011-01-11,GLD,-5.070120,-5.174413,-4.311878,-3.344378,1,0,0,0,...,0.006232,0.002946,0.000685,0.010444,0.006980,0.003172,-0.189073,0.922351,-0.811921,train
1,2011-01-12,GLD,-5.174413,-4.853728,-4.330529,-3.352055,2,0,0,0,...,0.006234,0.002946,0.000792,0.010357,0.006945,0.003130,2.027990,1.011525,0.279239,train
2,2011-01-13,GLD,-4.853728,-4.963198,-4.457583,-3.374136,3,0,0,0,...,0.006241,0.002943,0.000771,0.010345,0.006919,0.003113,0.666903,-0.005867,2.288776,train
3,2011-01-14,GLD,-4.963198,-5.450606,-4.434396,-3.371583,4,0,0,1,...,0.006241,0.002943,0.000712,0.010367,0.006911,0.003109,2.124020,0.849803,0.386406,train
4,2011-01-18,GLD,-5.450606,-5.419435,-4.368734,-3.374667,1,0,0,0,...,0.006262,0.002938,0.000713,0.010365,0.006900,0.003103,2.066790,0.059928,-0.236948,train


## 2. Materialize builder-compatible features
The HAR helper still expects the old-school `(date, asset, log_rv, rv)` parquet, so this cell reshapes the TFT-ready frame into that schema, fills in any naming drift, and writes `features.parquet` under `data/processed/`.

In [11]:
feat_path = DATA_PROCESSED / "features.parquet"

builder_df = tft_df.copy()
if "ticker" in builder_df.columns:
    builder_df = builder_df.rename(columns={"ticker": "asset"})
builder_df["log_rv"] = builder_df.get("logvol_t", builder_df.get("log_rv"))
if builder_df["log_rv"].isna().all():
    raise ValueError("TFT-ready data must have logvol_t or log_rv column.")
builder_df["log_rv"] = builder_df["log_rv"].astype(float)
builder_df["rv"] = np.exp(builder_df["log_rv"].clip(-50, 50))
builder_df[["date","asset","rv","log_rv"]].to_parquet(feat_path, index=False)
print(f"Wrote {feat_path} with {len(builder_df):,} rows from TFT-ready data.")
feat_path.exists()

Wrote /Users/sonalilonkar/Desktop/Projects/DL_Project/vol-forecasting/data/processed/features.parquet with 13,964 rows from TFT-ready data.


True

## 3. Load features and trim a window
To keep iteration tight I only look at the 2020–2021 window. This slice lets me inspect the `features.parquet` output quickly and make sure both SPY and TLT still have healthy coverage before calling the heavy bits.

In [16]:
import pandas as pd
subset_start = pd.Timestamp("2020-01-01")
subset_end = pd.Timestamp("2021-12-31")
try:
    feat_df = pd.read_parquet(feat_path)
except ImportError as exc:
    raise ImportError("Install pyarrow or fastparquet to read features.parquet") from exc
mask = (feat_df["date"] >= subset_start) & (feat_df["date"] <= subset_end)
feat_small = feat_df.loc[mask].copy()
print("Feature rows (subset)", len(feat_small))
display(feat_small.head())

Feature rows (subset) 2000


,date,asset,rv,log_rv
2243,2020-01-09,GLD,0.004697,-5.360777
2244,2020-01-10,GLD,0.002662,-5.928676
2245,2020-01-13,GLD,0.001830,-6.303171
2246,2020-01-14,GLD,0.003357,-5.696850
2247,2020-01-15,GLD,0.004170,-5.479871


### Quick ticker sanity check
After slicing to the pandemic window I double-check how many rows each ETF kept. If one ticker disappears, I know the upstream filters were too aggressive before I waste time modeling.

In [14]:
feat_small["asset"].value_counts()

asset
GLD    500
QQQ    500
SPY    500
TLT    500
Name: count, dtype: int64

## 4. Fit HAR baseline and emit predictions
This is the integration point with the real repo code: I hand the freshly-written parquet path to `fit_predict_har`, let it crank through its workflow, and capture the resulting prediction CSV under `experiments/preds/`. If this cell runs, the end-to-end wiring works.

In [17]:
from src.models.har_rv import fit_predict_har
pred_path = EXPERIMENTS / "preds" / "har_h1.csv"
fit_predict_har(feat_path=str(feat_path), out=str(pred_path))
pred_path.exists()

Wrote /Users/sonalilonkar/Desktop/Projects/DL_Project/vol-forecasting/experiments/preds/har_h1.csv with 13,876 rows.


True

## 5. Quick prediction diagnostics
Rather than trust the CSV blindly, I reload it, filter to the same window, and compute per-asset RMSE. The goal is to spot obvious pathologies (NaNs, zero variance, crazy errors) before calling any backtest.

In [18]:
pred_df = pd.read_csv(pred_path, parse_dates=["date"])
mask = (pred_df["date"] >= subset_start) & (pred_df["date"] <= subset_end)
pred_small = pred_df.loc[mask].copy()
pred_small["squared_error"] = (pred_small["y_true_logrv"] - pred_small["yhat_logrv"]) ** 2
summary = pred_small.groupby("asset").agg(
    rows=("date", "count"),
    rmse_logrv=("squared_error", lambda x: float((x.mean()) ** 0.5))
)
display(pred_small.head())
summary

,date,asset,y_true_logrv,y_true_rv,horizon,yhat_logrv,yhat_rv,model,squared_error
2221,2020-01-09,GLD,-5.360777,0.004697,1,-5.925615,0.002670,HAR-RV,0.319042
2222,2020-01-10,GLD,-5.928676,0.002662,1,-5.746239,0.003195,HAR-RV,0.033283
2223,2020-01-13,GLD,-6.303171,0.001830,1,-5.786318,0.003069,HAR-RV,0.267137
2224,2020-01-14,GLD,-5.696850,0.003357,1,-5.862810,0.002843,HAR-RV,0.027543
2225,2020-01-15,GLD,-5.479871,0.004170,1,-5.802941,0.003019,HAR-RV,0.104374


,rows,rmse_logrv
asset,,
GLD,500,0.405519
QQQ,500,0.412095
SPY,500,0.464419
TLT,500,0.400707


## 6. Backtest stub (weights/coverage)
Finally I push the trimmed predictions through the lightweight `backtest` helper. It only tracks weights, turnover, and basic coverage, but that’s enough to prove the HAR outputs line up with the downstream interface.

In [19]:
from src.backtest.engine import backtest
bt_out = backtest(pred_small)
bt_path = EXPERIMENTS / "results" / "har_h1_stub.csv"
bt_path.parent.mkdir(parents=True, exist_ok=True)
bt_out.to_csv(bt_path, index=False)
bt_out.tail()

,date,turnover,cost,port_sigma
495,2021-12-27,0.034759,0.000035,0.1
496,2021-12-28,0.095027,0.000095,0.1
497,2021-12-29,0.042842,0.000043,0.1
498,2021-12-30,0.033078,0.000033,0.1
499,2021-12-31,0.036180,0.000036,0.1


## 7. Notes & TODOs
- Make sure pyarrow stays installed so parquet I/O doesn’t fall over.
- Backfill realized returns + real portfolio P&L inside `src/backtest.engine`.
- Extend `har_rv.py` to cover h5/h22 and log results per horizon (maybe multi-index CSVs).
- Swap HAR with TFT once a Colab checkpoint exports predictions using the same schema.
- Automate this whole dry run via a Makefile target so it’s not a manual dance.